In [1]:
import os
import argparse
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn

In [4]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")            # Also you can set cloud URI path Mention in mlflow.md
mlflow.set_experiment("Wine-Quality01")

2026/05/28 11:13:41 INFO mlflow.tracking.fluent: Experiment with name 'Wine-Quality01' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1779947021146, experiment_id='4', last_update_time=1779947021146, lifecycle_stage='active', name='Wine-Quality01', tags={}, trace_location=None, workspace='default'>

In [5]:
def data_load(data_path:str) -> pd.DataFrame:

    try:
        df = pd.read_csv(data_path,sep=';')
        return df
    except Exception as e:
        raise e

In [6]:
def evaluate(y,pred):
    rmse = np.sqrt(mean_squared_error(y,pred))
    mae = mean_absolute_error(y,pred)
    r2 = r2_score(y,pred)

    return rmse, mae, r2

In [7]:
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()


In [8]:
import ssl
import urllib.request

# Create an unverified context
context = ssl._create_unverified_context()
response = urllib.request.urlopen("https://raw.githubusercontent.com/shrikant-temburwar/Wine-Quality-Dataset/refs/heads/master/winequality-red.csv", context=context)


In [9]:
df = data_load(data_path = "https://raw.githubusercontent.com/shrikant-temburwar/Wine-Quality-Dataset/refs/heads/master/winequality-red.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [10]:
train,test = train_test_split(df,random_state=42)
train_x = train.drop(["quality"],axis=1)
test_x = test.drop(["quality"],axis=1)

train_y = train[["quality"]]
test_y = test[["quality"]]

In [11]:
alpha=0.9
l1_ratio = 0.9

with mlflow.start_run():

    mlflow.set_tag("developer", "Dhaval Antala")
    mlflow.set_tag("Model", "elastic-net")

    mlflow.log_param('alpha', alpha)
    mlflow.log_param('l1_ratio', l1_ratio)

    lr = ElasticNet(alpha=alpha, l1_ratio=alpha)
    lr.fit(train_x,train_y)

    pred = lr.predict(test_x)

    rmse,mae,r2 = evaluate(test_y,pred)

    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('r2', r2)

    mlflow.sklearn.log_model(lr, 'ElasticNet')

    print(f"Elastic net Params: alpha: {alpha}, l1_ratio: {l1_ratio}")
    print(f"Elastic net metric: rmse:{rmse}, mae:{mae},r2:{r2}")

2026/05/28 11:13:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 11:13:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Elastic net Params: alpha: 0.9, l1_ratio: 0.9
Elastic net metric: rmse:0.7846134393282495, mae:0.645945835118589,r2:0.004810104895976441
🏃 View run unique-sow-161 at: http://127.0.0.1:5000/#/experiments/4/runs/38ae1242e57c4951a4ff21a74b41372e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [12]:
mlflow.autolog()

with mlflow.start_run():

    lr = ElasticNet(alpha=alpha, l1_ratio=alpha)
    lr.fit(train_x,train_y)

    pred = lr.predict(test_x)

    rmse,mae,r2 = evaluate(test_y,pred)

    print(f"Elastic net Params: alpha: {alpha}, l1_ratio: {l1_ratio}")
    print(f"Elastic net metric: rmse:{rmse}, mae:{mae},r2:{r2}")

2026/05/28 11:13:49 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/05/28 11:13:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Elastic net Params: alpha: 0.9, l1_ratio: 0.9
Elastic net metric: rmse:0.7846134393282495, mae:0.645945835118589,r2:0.004810104895976441
🏃 View run enthused-wolf-499 at: http://127.0.0.1:5000/#/experiments/4/runs/b036000a5a7a4b609fb0becb9d3c59ca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


## HYPERPARAMETER TUNING AND TRACKING USING MLFLOW

In [19]:
import mlflow
import mlflow.sklearn

from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split

In [20]:
# Load data
digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.2, random_state=42
)

In [21]:
mlflow.sklearn.autolog(max_tuning_runs=None)

In [22]:
# Define parameter grid
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
}

scoring_metrics = {
    "Accuracy": make_scorer(accuracy_score),
    "Precision": make_scorer(precision_score, average="macro", zero_division=0),
    "Recall": make_scorer(recall_score, average="macro"),
    "F1_Score": make_scorer(f1_score, average="macro")
}


In [ ]:
with mlflow.start_run():
    rf = RandomForestClassifier(random_state=42)

    grid_search = GridSearchCV(
        rf,
        param_grid,
        cv=5,
        scoring=scoring_metrics,
        refit="Accuracy", 
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("test_f1_macro", f1_score(y_test, y_pred, average="macro"))

    print(f"Best params: {grid_search.best_params_}")
    print(f"Best CV accuracy score: {grid_search.best_score_:.3f}")